# Retraining and Evaluation Loop
After a lot of modeling and data manipulation, the next step was to build a consistent loop to try models out and evaluate them.

This notebook precisely contains a way to train, evaluate, and save plots from our model's predictions against the ground truth on several different days. For each day, the model is retrained and used to forecast again every four hours.

## Library Imports

In [15]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
import statsmodels.api as sm
# From FPP3 book (The Pythonic Way)
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*FigureCanvasAgg is non-interactive.*"
)
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
from utilsforecast.plotting import plot_series as plot_series_utils
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})
import matplotlib as mpl
from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
from fpppy.utils import plot_series

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#569CC6", "#D55F03"],
        name='black_and_2color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55F03", "#569CC6", "#13A076"],
        name='black_and_3color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55F03", "#569CC6", "#13A076", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55F03", "#569CC6", "#13A076", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

In [19]:
import matplotlib.dates as matplotdates
import matplotlib.ticker as ticker
from functools import partial, reduce

import statsmodels.api as sm
from matplotlib.ticker import MaxNLocator
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.adapters.prophet import AutoARIMAProphet
from statsforecast.models import MSTL, AutoETS, AutoARIMA, ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.api import VAR
from utilsforecast.evaluation import evaluate
from utilsforecast.feature_engineering import trend, fourier, pipeline
from utilsforecast.losses import rmse, mae, mape, mase, smape
from utilsforecast.preprocessing import fill_gaps
import time
import holidays
import matplotlib.dates as mdates


In [20]:
from sklearn.preprocessing import StandardScaler
import time
import statsforecast

## Load the Data
These datasets have some modifications: They encode the day of the week as one-hot encodings.
But that's it. No Median Absolute Deviation yet, because the dataset is like a sliding window, and on each window you must recalculate the MAD.

In [1]:
import pandas as pd
import holidays
import os

os.makedirs("./data_apr", exist_ok=True)

# Load base data
ptc = pd.read_csv("./kafka_coding_2/data/ptc_weather.csv")
ptc["ds"] = pd.to_datetime(ptc["ds"])

# Add is_holiday for both years
ee_holidays_2022 = holidays.Estonia(years=[2022])
ee_holidays_2023 = holidays.Estonia(years=[2023])
all_holidays = {**ee_holidays_2022, **ee_holidays_2023}
ptc["is_holiday"] = ptc["ds"].dt.date.map(lambda x: 1 if x in all_holidays else 0)

# One-hot encode day_of_week (drop_first=True drops day 0, giving columns day_of_week_1 to day_of_week_6)
ptc_oh = pd.get_dummies(ptc, columns=["day_of_week"], drop_first=True)

# Split years
ptc_2022 = ptc_oh[ptc_oh["year"] == 2022].copy()
ptc_2023 = ptc_oh[ptc_oh["year"] == 2023].copy()

# Drop columns not needed for training
drop_cols = ["year", "month", "day", "hour", "in_sum", "out_sum"]

ptcw2022_oh = ptc_2022.drop(columns=drop_cols)
ptcw2022_oh.to_csv("./data_apr/ptcw2022_oh.csv", index=False)

ptc_weather_2023_ground_truth_oh = ptc_2023.drop(columns=drop_cols)
ptc_weather_2023_ground_truth_oh.to_csv("./data_apr/ptc_weather_2023_ground_truth_oh.csv", index=False)

# Dev dataset has no y — used as future exogenous regressors
ptc_weather_2023_dev_oh = ptc_2023.drop(columns=drop_cols + ["y"])
ptc_weather_2023_dev_oh.to_csv("./data_apr/ptc_weather_2023_dev_oh.csv", index=False)

print("Done. Datasets saved to ./data_apr/")
print(ptcw2022_oh.shape, ptc_weather_2023_dev_oh.shape, ptc_weather_2023_ground_truth_oh.shape)

Done. Datasets saved to ./data_apr/
(8733, 21) (8735, 20) (8735, 21)


In [2]:
# 2022 dataset
ptcw2022_oh = pd.read_csv("./data_apr/ptcw2022_oh.csv")
# 2023 development dataset, doesn't contain "y"
ptc_weather_2023_dev_oh = pd.read_csv("./data_apr/ptc_weather_2023_dev_oh.csv")
# 2023 Ground Truth dataset, serves to make the "sliding window" training dataset.
ptc_weather_2023_ground_truth_oh = pd.read_csv("./data_apr/ptc_weather_2023_ground_truth_oh.csv")

# Let's make the "ds" column, which contains the timestamps, be of type datetime.
ptcw2022_oh["ds"] = pd.to_datetime(ptcw2022_oh["ds"])
ptc_weather_2023_dev_oh["ds"] = pd.to_datetime(ptc_weather_2023_dev_oh["ds"])
ptc_weather_2023_ground_truth_oh["ds"] = pd.to_datetime(ptc_weather_2023_ground_truth_oh["ds"])

## Data-preparing functions
These functions are used to prepare the new dataset we use at each iteration

In [3]:
def scale_datasets(train_df, test_df, gt_df):
    """ Every loop the sliding window means you get a new dataset, meaning you gotta refit the data.
        This method takes the train dataset to train a scaler and scale the data.
        Then we reuse that scaler on the test and ground-truth dataset
        (Though we don't use the ground-truth dataset in that manner)

        Arguments:
            train_df: pd.DataFrame - Training Dataset.
            test_df: pd.DataFrame - Dev or Testing Dataset
            gt_df: pd.DataFrame - Ground Truth dataframe (does contain "y")

        Returns:
            train_df_local: pd.DataFrame - The Training dataset, scaled with the Scikit-Learn scaler
            test_df_local: pd.DataFrame - The dev dataset, scaled using the training set fit of the scaler
            gt_df_local: pd.DataFrame - The ground truth dataframe, scaled (though really not needed)
    """

    # These are the columns to scale.
    weather_cols = [
        'temp_mean', 'humidity_mean', 'precipitation_mean', 'rain_mean', 
        'snowfall_mean', 'snow_depth_mean', 'wind_speed_mean', 'cloud_cover_mean',
        'cloud_cover_low_mean', 'cloud_cover_mid_mean', 'cloud_cover_high_mean'
    ]
    train_df_local = train_df.copy()
    
    scaler = StandardScaler()
    # Fit and scale local
    train_df_local[weather_cols] = scaler.fit_transform(train_df_local[weather_cols])
    
    # Scale dev data using the same training fit
    test_df_local = test_df.copy()
    test_df_local[weather_cols] = scaler.transform(test_df_local[weather_cols])

    # We also applied it to the Ground Truth data just in case, but it was not needed after all
    gt_df_local = gt_df.copy()
    gt_df_local[weather_cols] = scaler.transform(gt_df_local[weather_cols])
    return train_df_local, test_df_local, gt_df_local
    
def rescale_using_MAD(train_df, multiplier=3):
    """ Every training set we gotta apply the Median Absolute Deviation as well.
        This dataset obtains the Median, then applies multiplier*MAD threshold

        Arguments:
            train_df: pd.DataFrame - The training dataset to be rescaled using MAD
            multiplier : int - The threshold of MAD to use (3 is what we use)

        Returns:
            train_df_mad : pd.DataFrame - The training dataset adjusted.
    """
    train_df_mad = train_df.copy()
    # Step 1: Calculate the Median for the time series
    median = train_df_mad["y"].median()
    # Step 2: Calculate the median absolute deviation
    mad = (train_df_mad["y"] - median).abs().median()
    # Step 3: The threshold is X times that
    threshold = multiplier * mad
    # Then we simply find those values and set it to the median
    train_df_mad.loc[
        (train_df_mad["y"] - median).abs() > threshold, #Wherever this is True gets selected
        "y"
    ] = median
    return train_df_mad

def create_train_dev_splits(train_df, dev_df, gt_df, start_date : pd.Timestamp, window_size : int):
    """ Will create the correct split given the 2022, 2023 data.
        We have to take the data from 2023 that is before the current start_date
        and the new dev set must have only after the start_date

        Arguments:
            train_df : pd.DataFrame - The Training dataframe
            dev_df : pd.DataFrame - The Testing dataframe
            gt_df : pd.DataFrame - The Ground Truth dataframe (has "y")
            start_date : pd.Timestamp - The timestamp on which to readjust the datasets
            window_size: int - How many entries to take

        Returns:
            pass_gt : pd.DataFrame - The new training dataframe. Must still be scaled.
            dev_dataset : pd.DataFrame - The testing dataframe, with only dates after the start date
            gt_dataset : pd.DataFrame - The ground truth dataframe, with only dates after the startd ate
            
    """

    # Let's get all dates from 2023 that are strictly before the start date
    pass_gt_dev = gt_df[
        gt_df["ds"] < start_date
    ]

    # Now let's slide our window from 2022, into the future.
    window_train = window_size - len(pass_gt_dev)

    # This creates the new training dataset: the old entries (removing some) + the new entries from 2023
    pass_gt = pd.concat([
        train_df.tail(window_train),
        pass_gt_dev
    ])

    # Now the dev dataset must be reduced to only dates after start_date
    dev_dataset = dev_df[
        dev_df["ds"] >= start_date
    ]

    # Same goes for our ground_truth. We use this to plot in red the real values.
    gt_dataset = gt_df[
        gt_df["ds"] >= start_date
    ]
    
    return pass_gt, dev_dataset, gt_dataset

## Functions to train the models and forecast with them
The loop of splitting the dataset, then training and forecasting, then saving the plots is very similar.
These are to help us replace the model we're using in the loop easily.

In [4]:
# Functions to train Models. Each one has different requirements, after all.
def create_pmodel():
    """ Function to create a Prophet Model without fitting it.

        Returns:
            pmodel - A newly-instanced Prophet model without training
    """
    pmodel = Prophet(
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="additive",
        changepoint_prior_scale = 0.1 #Controls model flexibility.
        # It is about letting the model pursue short-term trend shifts.
        # 0.05 is the default, 0.001 is very strict (doesn't allow it).
        # So we chose 0.1 to try and give it some flexibility.
    )
    # Add the exogenous variables
    pmodel.add_regressor("temp_mean")
    pmodel.add_regressor("humidity_mean")
    pmodel.add_regressor("precipitation_mean")
    pmodel.add_regressor("rain_mean")
    pmodel.add_regressor("snowfall_mean")
    pmodel.add_regressor("snow_depth_mean")
    pmodel.add_regressor("wind_speed_mean")
    pmodel.add_regressor("cloud_cover_mean")
    pmodel.add_regressor("cloud_cover_low_mean")
    pmodel.add_regressor("cloud_cover_mid_mean")
    pmodel.add_regressor("cloud_cover_high_mean")
    # Prophet actually has a built-in way to add holidays
    pmodel.add_country_holidays(country_name='EE')#pmodel.add_regressor("is_holiday")
    return pmodel


def train_forecast_MSTL_ARIMA(train_df, test_df, horizon_window):
    """ Trains an MSTL model with ARIMA on the Residuals + Trend (+ Exogenous)

        Arguments:
            train_df : pd.DataFrame - The training dataset, ready to use
            test_df : pd.DataFrame - The testing dataset, ready to use
            horizon_window : int - How many hours to predict in the future (we set it to 8 hours)

        Returns:
            moving_forecasts : pd.DataFrame - The model's forecasts for the next X hours.
    """

    # Create the model: MSTL(w,d,) + AutoARIMA + Exogenous
    sf = StatsForecast(
        models=[MSTL(
            season_length=[24, 24*7], #Daily and Weekly Seasonality
            trend_forecaster=AutoARIMA()
        )],
        freq='1h',
        n_jobs=1,  # start with 1 to avoid multiprocessing issues
    )

    # The forecast() function fits the model to the train_df then predicts.
    moving_forecasts = sf.forecast(
        h=horizon_window,
        df=train_df,
        X_df = test_df,
        level=[80, 95]
    )
    return moving_forecasts

def train_forecast_Prophet_ARIMA(train_df, test_df, horizon_window):
    """ Will train a Prophet model with an ARIMA error model

        Arguments:
            train_df : pd.DataFrame - The training dataset ready to use
            test_df : pd.DataFrame - The testing dataset ready to use
            horizon_window : int - How many hours to predict into the future (we set it to 8 hours)

        Returns:
            fpa_preds: pd.DataFrame - The dataframe with the model's predictions
    """

    # STEP 0: We create a Prophet model
    pmodel_arima = create_pmodel()
    # STEP 1: We fit the model on the training dataset
    pmodel_arima.fit(
        train_df
    )
    
    # STEP 2: PREDICT ON THE TRAINING DATASET
    pforecast_train = pmodel_arima.predict(
        train_df
    )
    
    # STEP 3: GET THE RESIDUALS (The error the models commit)
    # First we merge the training dataframe with Prophet's predictions
    prophet_residuals = train_df.merge(
        pforecast_train,
        on="ds",
        how="left"
    )
    # Then we calculate the residuals in a new column
    prophet_residuals["residuals"] = prophet_residuals["y"] - prophet_residuals["yhat"]
    # Lastly, we take from this dataframe only the timestamp ("ds") and the "residuals"
    prophet_r = prophet_residuals[["ds", "residuals"]].copy().rename(
        columns={
            "residuals":"y"
        }
    )
    # And to train an ARIMA model, we need to create a column named "unique_ds", as always.
    prophet_r["unique_id"] = "all_sensors"
    
    # STEP 4: TRAIN AN ARIMA MODEL ON THE RESIDUALS
    # This order I chose after training once, I checked the coefficients
    # and changed AutoARIMA to this, since it's faster.
    prophet_arima = StatsForecast(
        models=[
            ARIMA(
                order=(2,0,1),
                season_length=1
            )
        ],
        freq="1h"
    )
    # We fit the ARIMA model
    prophet_arima_fit = prophet_arima.fit(prophet_r)
    
    # STEP 5: PREDICT WITH PROPHET on the Testing Dataset
    pforecast_dev = pmodel_arima.predict(
        test_df[
        ["ds", 'temp_mean', 'humidity_mean', 'precipitation_mean', 'rain_mean', 
        'snowfall_mean', 'snow_depth_mean', 'wind_speed_mean', 'cloud_cover_mean',
        'cloud_cover_low_mean', 'cloud_cover_mid_mean', 'cloud_cover_high_mean']
        ].head(horizon_window)
    )
    pforecast_dev["unique_id"] = "all_sensors"
    # We rename Prophet's predictions column from "yhat" to "y"
    pfdf = pforecast_dev[['ds', 'yhat']].copy().rename(columns={'yhat': 'y'})
    pfdf['unique_id'] = 'all_sensors'
    
    # STEP 6: PREDICT WITH ARIMA on the Testing Dataset
    parima_preds = prophet_arima_fit.predict(
        h=horizon_window,
        level=[80, 95]
    )
    
    # STEP 7: JOIN PREDICTIONS from Prophet and from ARIMA
    final_prophet_arima_preds = pfdf[["ds", "y"]].merge(
        parima_preds[["ds", "ARIMA"]],
        on="ds",
        how="inner"
    )
    # Now we use this merged DataFrame to add up Prophet's and ARIMA's predictions
    final_prophet_arima_preds["yhat_final"] = final_prophet_arima_preds["y"] + final_prophet_arima_preds["ARIMA"]
    final_prophet_arima_preds["unique_id"] = "all_sensors"
    
    fpa_preds = final_prophet_arima_preds[["unique_id", "ds", "yhat_final"]].copy().rename(
        columns={
            "yhat_final":"ARIMA"
        }
    )
    return fpa_preds

def train_forecast_Fourier_ARIMA(train_df, test_df, horizon_window):
    """ Trains an ARIMA model with Fourier Terms (k=2)

        Arguments:
            train_df : pd.DataFrame - The training dataset, ready to use
            test_df : pd.DataFrame - The testing dataset, ready to use
            horizon_window : int - How many hour sto predict in the future (We set it to 8)

        Returns:
            fc_arimaf : pd.DataFrame - The model's forecasting predictions
    """
    # Step 1: Build the Fourier Terms. "k" can be tuned.
    fourier_daily = partial(
            fourier,
            season_length = 24, k=2 #Daily, k=2
    )
    fourier_weekly = partial(
        fourier,
        season_length = 24*7, k=2 #Weekly, k=2
    )
    
    # Step 2: Build the pipeline object. Taken from the book, this yields new datasets.
    fourier_training_dataset, fourier_forecasting_X_df = pipeline(
        train_df,
        features=[
            fourier_daily, fourier_weekly
        ],
        freq="1h",
        h=horizon_window
    )
    # Step 3: For the testing dataset, we need to merge Fourier Terms with the original dev dataset.
    fourier_forecasting_X_df = fourier_forecasting_X_df.merge(
        test_df[[
            'day_of_week_1','day_of_week_2','day_of_week_3','day_of_week_4','day_of_week_5','day_of_week_6',
            'temp_mean', 'humidity_mean', 'precipitation_mean',
           'rain_mean', 'snowfall_mean', 'snow_depth_mean', 'wind_speed_mean',
           'cloud_cover_mean', 'cloud_cover_low_mean', 'cloud_cover_mid_mean',
           'cloud_cover_high_mean', 'ds', 'is_holiday'
        ]],
        on="ds",
        how="inner"
    )

    # Step 4: Create the Fourier+ARIMA dataset.
    arimaf = StatsForecast(
        models=[
            ARIMA(
                order=(2,0,4),#p=2,q=4,d=0,
                season_length=1
            )
        ],
        freq="1h"
    )
    # Step 5: Fit the model
    arimaf_fit = arimaf.fit(
        fourier_training_dataset
    )
    # Step 6: Predict with the model
    fc_arimaf = arimaf_fit.predict(
        h=horizon_window,
        X_df=fourier_forecasting_X_df.head(horizon_window),
        level=[80, 95]
    )
    return fc_arimaf

## Plotting Function
A reusable function to just create a plot and save it into a png.

In [5]:
def save_plot(train_df, gt_df, preds_df, model: str, timestamp, horizon_window):
    """ Creates a plot and saves it into the img_eval folder. 
    
        Arguments:
            train_df : pd.DataFrame - The training dataset, ready to use
            gt_df : pd.DataFrame - The ground truth dataset, ready to use, to show the actual ground truth values
            model : str - The model we're going to use. Helps us know which column to use, how to name the file, and how to title the plot.
            timestamp : pd.Timestamp - The date we're plotting at, used to save the file with the date as indicator
            horizon_window : int - How many hours into the future we're predicting (to show ground truth)

        Returns:
            Nothing
    """
    # The variables we vary on each plot. This allows us to reuse the plotting code.
    model_config = {
        "mstl":    ("MSTL",  "mstl",   "MSTL(d, w) + AutoARIMA + Exogenous",        {"level": [80, 95]}),
        "fourier": ("ARIMA", "farima",  "Fourier(d, w, k=2) + ARIMA + Exogenous",    {}),
        "prophet": ("ARIMA", "parima",  "Prophet (w Regressors) + ARIMA",             {"level": [80, 95]}),
    }
    # Model col in the DataFrame, Model Prefix to use when saving, Title to use on the Plot, and Levels to plot Confidence Interval
    model_col, prefix, title, extra_kwargs = model_config[model]

    # This part's taken from the book as well, we adjusted it.
    fig, ax = plt.subplots(figsize=(16, 5))
    plot_series(
        df=train_df,
        forecasts_df=preds_df,
        models=[model_col], #Tells the model which columns to plot, really.
        max_insample_length=12,
        xlabel="DateTime [1h]", ylabel="People Count",
        title=f"Hourly foot traffic in Tallinn ({title})",
        palette="black_and_blue", rm_legend=False, ax=ax, #Specify ax=ax here to use the ax object above to plot
        **extra_kwargs
    )
    # Setting the plot's ticks
    ax.xaxis.set_major_locator(mdates.HourLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    # Plotting the real data agains the forecasts.
    ax.plot(gt_df["ds"].head(horizon_window), gt_df["y"].head(horizon_window), color="red", alpha=0.5)
    # We save the figure to the folder, so that we can check all iterations at the end.
    fig.savefig(f'img_eval/{prefix}_{timestamp.strftime("%Y-%m-%d_%H-%M")}.png', bbox_inches="tight", dpi=150)
    plt.close(fig)

In [ ]:
def train_per_day(train_df, dev_df, gt_df, timestamp_list, max_iterations, hours_btw_forecasts, model : str, hour_window = 8):
    """ Will train a model at a timestamp multiple times, in a rolling-window fashion. Aggregates evaluations.

        Arguments:
            train_df : pd.DataFrame - The training dataset (complete, gets adjusted)
            dev_df : pd.DataFrame - The testing dataset (complete, gets adjusted)
            gt_df : pd.DataFrame - The ground truth dataset (complete, gets adjusted)
            timestamp_list : List[pd.Timestamp] - The dates on which to train the model (each day is trained every 6 times, because we train every 4 hours)
            max_iterations : int - THe number of iterations on a single day (6 times here)
            hours_btw_forecasts : int - The hours to slide the window between iterations (4 hours here)
            model : str - Either "mstl", "prophet", or "fourier"
            hour_window : int - How many hours into the future to predict.
    """
    # Accumulator for model predictions
    all_model_preds = []
    # For each timestamp we will use to evaluate our model...
    for ts in timestamp_list:
        print(f"!!! Starting on Date: {ts.date()}")

        # We will retrain our model a total of max_iterations (6 times)
        for i in range(0, max_iterations, 1):
            
            # Since model trains every 4 hours, this simulates that.
            loop_start_date = ts + pd.Timedelta(hours=i*hours_btw_forecasts)
            print(f"-->Starting on dt: {loop_start_date.strftime('%Y-%m-%d_%H-%M')}")

            # Step 2: Create the datasets for this loop.
            loop_train_df, loop_dev_df, loop_gt_df = create_train_dev_splits(
                train_df=train_df,
                dev_df=dev_df,
                gt_df=gt_df,
                start_date=loop_start_date,
                window_size=len(train_df)
            )

            # Step 3: Rescale this loop's training DF using MAD
            loop_train_df = rescale_using_MAD(
                loop_train_df,
                multiplier=3
            )
            # Step 4: Rescale this loops' training DF (Exogenous variables)
            loop_train_df, loop_dev_df, loop_gt_df = scale_datasets(
                loop_train_df,
                loop_dev_df,
                loop_gt_df
            )

            # Step 5: Fit a model and return model predictions
            if model == "mstl":
                model_preds = train_forecast_MSTL_ARIMA( #Has columns ['unique_id', 'ds', 'MSTL', 'MSTL-lo-95', 'MSTL-lo-80', 'MSTL-hi-80', 'MSTL-hi-95']
                    loop_train_df,
                    loop_dev_df.head(hour_window),
                    horizon_window=hour_window
                )

                # below code clips the prediction value by 5th perccentile, this helped improve the model a little bit, the ratioanle 
                # behind this that the model shouldn't predict very low values because it is unlikely that tallinn would have such a low value of total count
                # other than a few times, this is evident from the data, basically we are limiting the lower band of confidence interval of our model.
                floor_value = loop_train_df["y"].quantile(0.05)
                model_preds["MSTL"] = model_preds["MSTL"].clip(lower=floor_value)
                model_preds["MSTL-lo-80"] = model_preds["MSTL-lo-80"].clip(lower=floor_value)
                model_preds["MSTL-lo-95"] = model_preds["MSTL-lo-95"].clip(lower=floor_value)

            elif model == "prophet":
                model_preds = train_forecast_Prophet_ARIMA( #Cols are: "unique_id", "ds", and "ARIMA"
                    loop_train_df,
                    loop_dev_df,
                    horizon_window=hour_window
                )
            else: #model == "fourier"
                model_preds = train_forecast_Fourier_ARIMA( #Cols are ['unique_id', 'ds', 'ARIMA']
                    loop_train_df,
                    loop_dev_df,
                    horizon_window=hour_window
                )
            model_preds['iteration'] = i
            model_preds['cutoff'] = loop_start_date
            # Step Extra: Naive Error for Evaluation later (MASE)
            naive_errors = loop_train_df["y"].diff(24).abs().dropna()
            naive_mae = naive_errors.mean()
            model_preds['naive_mae'] = naive_mae

            # Storing model results
            all_model_preds.append(model_preds)

            # Step 6: Save this loop's prediction plot
            save_plot(
                train_df=loop_train_df,
                gt_df=loop_gt_df,
                preds_df=model_preds,
                model=model,
                timestamp=loop_start_date,
                horizon_window=hour_window
            )

    # Step 7: Evaluate and save
    all_preds_df = pd.concat(all_model_preds, ignore_index=True)
    # Get the ground truth to compare to...
    actual_values = gt_df[['unique_id', 'ds', 'y']].copy()
    
    # Then merge it to the predictions.
    eval_df = all_preds_df.merge(actual_values, on=["unique_id", 'ds'], how='left') #unique_id here is not really needed but anyway.
    
    # Now to compute the metrics....
    if model == "mstl":
        eval_df['error'] = eval_df["MSTL"] - eval_df['y']
        model_col = "MSTL"
    elif model == "prophet":
        eval_df["error"] = eval_df["ARIMA"] - eval_df["y"]
        model_col = "ARIMA"
    else: #model == "fourier"
        eval_df["error"] = eval_df["ARIMA"] - eval_df["y"]
        model_col = "ARIMA"
        
    eval_df['abs_error'] = eval_df['error'].abs()
    eval_df['mase'] = eval_df['abs_error'] / eval_df['naive_mae']
    print("MAE: ", eval_df['abs_error'].mean())
    print("RMSE:", (eval_df['error']**2).mean()**0.5)
    print("sMAPE:", (2 * eval_df['abs_error'] / (eval_df[model_col].abs() + eval_df['y'].abs())).mean())
    print("MASE:", eval_df['mase'].mean())
    
    return eval_df

## Setting up the Timestamps to use

In [8]:
# The function above will loop over these timestamps. They were chosen to avoid holidays
eval_timestamps = [
    pd.Timestamp("2023-01-16 00:00:00"),
    pd.Timestamp("2023-02-13 00:00:00"),
    pd.Timestamp("2023-03-13 00:00:00"),
    pd.Timestamp("2023-04-17 00:00:00"),
    pd.Timestamp("2023-05-15 00:00:00"),
    pd.Timestamp("2023-06-12 00:00:00"),
    pd.Timestamp("2023-07-17 00:00:00"),
    pd.Timestamp("2023-08-14 00:00:00"),
    pd.Timestamp("2023-09-11 00:00:00"),
    pd.Timestamp("2023-10-16 00:00:00"),
    pd.Timestamp("2023-11-13 00:00:00"),
    pd.Timestamp("2023-12-11 00:00:00"),
]

## Training MSTL
This one is by far the slowest.

In [ ]:
start_mstl = time.time()
mstl_eval_df = (
    train_df=ptcw2022_oh,
    dev_df=ptc_weather_2023_dev_oh,
    gt_df=ptc_weather_2023_ground_truth_oh,
    timestamp_list=eval_timestamps,
    max_iterations=6,
    hours_btw_forecasts=4,
    model = "mstl",
    hour_window = 8
)
end_mstl = round(time.time() - start_mstl, 2)

!!! Starting on Date: 2023-01-16
-->Starting on dt: 2023-01-16_00-00
-->Starting on dt: 2023-01-16_04-00
-->Starting on dt: 2023-01-16_08-00
-->Starting on dt: 2023-01-16_12-00
-->Starting on dt: 2023-01-16_16-00
-->Starting on dt: 2023-01-16_20-00
!!! Starting on Date: 2023-02-13
-->Starting on dt: 2023-02-13_00-00
-->Starting on dt: 2023-02-13_04-00
-->Starting on dt: 2023-02-13_08-00
-->Starting on dt: 2023-02-13_12-00
-->Starting on dt: 2023-02-13_16-00
-->Starting on dt: 2023-02-13_20-00
!!! Starting on Date: 2023-03-13
-->Starting on dt: 2023-03-13_00-00
-->Starting on dt: 2023-03-13_04-00
-->Starting on dt: 2023-03-13_08-00
-->Starting on dt: 2023-03-13_12-00
-->Starting on dt: 2023-03-13_16-00
-->Starting on dt: 2023-03-13_20-00
!!! Starting on Date: 2023-04-17
-->Starting on dt: 2023-04-17_00-00
-->Starting on dt: 2023-04-17_04-00
-->Starting on dt: 2023-04-17_08-00
-->Starting on dt: 2023-04-17_12-00
-->Starting on dt: 2023-04-17_16-00
-->Starting on dt: 2023-04-17_20-00
!!! 

## Training Prophet+ARIMA
This one is surprisingly quick!

In [ ]:
start_prophet = time.time()
prophet_eval_df = train_per_day(
    train_df=ptcw2022_oh,
    dev_df=ptc_weather_2023_dev_oh,
    gt_df=ptc_weather_2023_ground_truth_oh,
    timestamp_list=eval_timestamps,
    max_iterations=6,
    hours_btw_forecasts=4,
    model = "prophet",
    hour_window = 8
)
end_prophet = round(time.time() - start_prophet, 2)

## Training Fourier+ARIMA
This one is quick if you use a fixed ARIMA. But if you want to try AutoARIMA to find something different, it might become slow

In [ ]:
start_fourier = time.time()
fourier_eval_df = train_per_day(
    train_df=ptcw2022_oh,
    dev_df=ptc_weather_2023_dev_oh,
    gt_df=ptc_weather_2023_ground_truth_oh,
    timestamp_list=eval_timestamps,
    max_iterations=6,
    hours_btw_forecasts=4,
    model = "fourier",
    hour_window = 8
)
end_fourier = round(time.time() - start_fourier, 2)

## Summarizing Evaluations
This function's just to print what we got

In [ ]:
## Here you can load the model results we got in case you want to jump right into it.
mstl_results_df = pd.read_csv("./eval_results/mstl_results.csv")
prophet_results_df = pd.read_csv("./eval_results/prophet_results.csv")
fourier_results_df = pd.read_csv("./eval_results/fourier_results.csv")

In [ ]:
def summarize_eval(eval_df, model_col, name):
    """ Will display a model's results using its evaluation DataFrame

        Argumgents:
            eval_df : pd.DataFrame - The dataframe that stores the model's evaluation metrics per iteration
            model_col : str - The string for the column where we store model predictions.
            name : str - The name of the model, so that the print shows that.
    """
    
    error = eval_df['error']
    abs_error = eval_df['abs_error']
    mase = eval_df['mase']
    pred = eval_df[model_col]
    y = eval_df['y']
    
    print(f"=== {name} Summary ===")
    print(f"MAE:   {abs_error.mean():.2f}")
    print(f"RMSE:  {(error**2).mean()**0.5:.2f}")
    print(f"sMAPE: {(2 * abs_error / (pred.abs() + y.abs())).mean():.4f}")
    print(f"MASE:  {mase.mean():.4f}")
    print()
    # Here we group per the "iteration" variable, so dates get mixed into the time periods
    print("--- Per Iteration ---")
    print(eval_df.groupby('iteration')[['abs_error', 'mase']].mean().round(3))
    # Here we group by date, so the iterations on one date are all combined.
    print("--- Per date ---")
    eval_df['date'] = pd.to_datetime(eval_df['cutoff']).dt.date
    print(eval_df.groupby('date')[['abs_error', 'mase']].mean().round(3))
    # Here we group by "cutoff", so it's the most granular of the evaluations.
    print("-- Per cutoff ---")
    print(eval_df.groupby('cutoff')[['abs_error', 'mase']].mean().round(3))
    print()
    print("================================")

# Run it for each model
summarize_eval(mstl_eval_df, "MSTL", "MSTL")
summarize_eval(prophet_eval_df, "ARIMA", "Prophet+ARIMA")
summarize_eval(fourier_eval_df, "ARIMA", "Fourier+ARIMA")